In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

from dataset_attrs import DATASET_ATTRS
from learner import MODEL_ATTRS
from distillation import run_embedding_distillation


In [ ]:
MODEL_NAMES = list(MODEL_ATTRS.keys())
DATASETS = list(DATASET_ATTRS.keys())
DPCS = [1, 3, 5, 10, 20]
LABEL_TYPES = ["hard", "soft"]
SKIP_COMPLETED = True

# Smoke defaults. Set both to None for full runs.
MAX_TRAIN_BATCHES_PER_EPOCH = 5
MAX_EVAL_BATCHES = 5

COMMON = dict(
    seq_length=128,
    inner_steps=1,
    train_epochs=1,
    batch_size_per_label=None,
    attention_label_type="none",
    train_batch_size=32,
    eval_batch_size=128,
    max_train_batches_per_epoch=MAX_TRAIN_BATCHES_PER_EPOCH,
    max_eval_batches=MAX_EVAL_BATCHES,
    n_eval_model=1,
    bf16=torch.cuda.is_available(),
    data_root=ROOT / "data",
)

RESULTS_DIR = ROOT / "results" / "embedding_distillation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS, MODEL_NAMES, DPCS, LABEL_TYPES


In [ ]:
rows = []

for task_name in DATASETS:
    for model_name in MODEL_NAMES:
        model_slug = model_name.replace('/', '_')
        for label_type in LABEL_TYPES:
            for dpc in DPCS:
                run_dir = RESULTS_DIR / task_name / f"{model_slug}_{label_type}_dpc_{dpc}"
                summary_path = run_dir / "summary.json"
                if SKIP_COMPLETED and summary_path.exists():
                    rows.append(json.loads(summary_path.read_text()))
                    continue
                synthetic_data, metrics = run_embedding_distillation(
                    task_name=task_name,
                    model_name=model_name,
                    dpc=dpc,
                    label_type=label_type,
                    save_dir=run_dir,
                    seed=42,
                    **COMMON,
                )
                row = {
                    "task_name": task_name,
                    "model_name": model_name,
                    "label_type": synthetic_data.config.label_type,
                    "dpc": dpc,
                    "num_synthetic_examples": synthetic_data.num_examples,
                    "seq_length": synthetic_data.config.seq_length,
                    "hidden_size": synthetic_data.hidden_size,
                    "inner_steps": synthetic_data.train_config.inner_steps,
                    **metrics,
                }
                rows.append(row)
                with open(summary_path, "w") as f:
                    json.dump(row, f, indent=2)

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary


In [ ]:
# Optional: one custom run outside the full grid.
# synthetic_data, metrics = run_embedding_distillation(
#     task_name="ag_news",
#     model_name="bert-base-uncased",
#     dpc=1,
#     save_dir=RESULTS_DIR / "ag_news_bert_base_hard_dpc_1_custom",
#     **{**COMMON, "label_type": "hard"},
# )


In [ ]:
# Optional next step: CLS attention labels. This is substantially heavier.
# synthetic_data, metrics = run_embedding_distillation(
#     task_name="ag_news",
#     model_name="bert-base-uncased",
#     dpc=1,
#     save_dir=RESULTS_DIR / "ag_news_attention_cls_dpc_1",
#     **{**COMMON, "attention_label_type": "cls"},
# )
